# Synthetic Data Generator
### Step 6.5: Kafka Producer Queue Manager Testing

In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
    execute_sql,
)

from utils.synthetic.pipeline.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    read_simulation_state_control,
    get_send_queue_status_counts,
    claim_pending_sensor_messages_batch,
    mark_claimed_batch_sent,
    mark_claimed_batch_failed,
    requeue_failed_messages,
    release_stale_claims,
)

from utils.core.env_helpers import (
    env_float, 
    env_int, 
    env_str,
)

In [ ]:
# -----------------------------------------------------------------------------
# 6.5 Testing / Sandbox Safety Flags
# -----------------------------------------------------------------------------
# This notebook is NOT part of the official pipeline.
# These flags intentionally default to safe/no-op behavior.

RUN_CLAIM_TEST_FLAG = True

# Reset claimed-but-unsent rows back to pending.
# This is usually safe for test claims.
RESET_TEST_CLAIMS_FLAG = True

# Optional lifecycle tests.
# Leave these False unless you are intentionally testing that transition.
MARK_SENT_TEST_FLAG = False
MARK_FAILED_TEST_FLAG = False
REQUEUE_FAILED_TEST_FLAG = False
RELEASE_STALE_CLAIMS_TEST_FLAG = False

# Emergency reset should almost always stay False.
# It can alter larger portions of the queue.
EMERGENCY_RESET_ALL_UNSENT_CLAIMS_FLAG = False

TEST_ERROR_MESSAGE = "Manual Stage 6.5 failed-message test"

print("Stage 6.5 sandbox flags")
print("run claim test:", RUN_CLAIM_TEST_FLAG)
print("reset test claims:", RESET_TEST_CLAIMS_FLAG)
print("mark sent test:", MARK_SENT_TEST_FLAG)
print("mark failed test:", MARK_FAILED_TEST_FLAG)
print("requeue failed test:", REQUEUE_FAILED_TEST_FLAG)
print("release stale claims test:", RELEASE_STALE_CLAIMS_TEST_FLAG)
print("emergency reset all unsent claims:", EMERGENCY_RESET_ALL_UNSENT_CLAIMS_FLAG)

In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

SIMULATION_TABLE_NAME = env_str(
    "SYNTHETIC_CONTROL_TABLE",
    "simulation_state_control",
    aliases=("PRODUCER_CONTROL_TABLE",),
)

SEND_QUEUE_TABLE_NAME = env_str(
    "SYNTHETIC_SEND_QUEUE_TABLE",
    "synthetic_sensor_messages_send_queue",
    aliases=("PRODUCER_QUEUE_TABLE",),
)

PRODUCER_TOPIC = env_str(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
    aliases=("KAFKA_TOPIC",),
)

PRODUCER_WORKER_ID = env_str(
    "SYNTHETIC_PRODUCER_WORKER_ID",
    "producer_worker_test_001",
)

OBSERVATION_BATCH_SIZE = env_int("OBSERVATION_BATCH_SIZE", 500)
PRODUCER_POLL_SECONDS = env_float("PRODUCER_POLL_SECONDS", 0.0)
PRODUCER_MAX_SEND_ATTEMPTS = env_int("PRODUCER_MAX_SEND_ATTEMPTS", 3)

CONTROL_OWNER_ROLE = "kafka_producer"
APPLY_OWNER_AND_GRANTS_FLAG = False


print("Producer queue manager claim/reset test config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("send queue table:", SEND_QUEUE_TABLE_NAME)
print("producer topic:", PRODUCER_TOPIC)
print("producer worker id:", PRODUCER_WORKER_ID)
print("producer max send attempts:", PRODUCER_MAX_SEND_ATTEMPTS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)

Producer queue manager test config
schema: capstone
send queue table: synthetic_sensor_messages_send_queue
producer topic: pump.telemetry.synthetic
producer worker id: producer_worker_test_001
number of sensors: 52
observation batch size: 500
message batch size: 26000


---

In [7]:

engine = get_engine_from_env()


---

In [ ]:
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    ).loc[0, "sensor_count"]
)

PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS

print("Derived claim batch sizing")
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("producer batch size:", PRODUCER_BATCH_SIZE)

---

In [ ]:
control_row = read_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)

display(control_row)

---

In [ ]:
ensure_send_queue_runtime_columns(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Ensured send queue runtime columns and indexes.")

---

In [ ]:
queue_health_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS unclaimed_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS claimed_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_timestamp_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(queue_health_dataframe)

,queue_status,row_count,unclaimed_count,claimed_count,unsent_count,sent_timestamp_count
0,pending,11700000,11700000,0,11700000,0


---

In [ ]:
# -----------------------------------------------------------------------------
# Stage 7 readiness helper
# -----------------------------------------------------------------------------
# Safe diagnostic only. This does not claim, reset, mark sent, or mutate rows.

def check_stage_7_readiness(
    engine,
    *,
    schema: str,
    send_queue_table_name: str,
    dataset_id: str,
    run_id: str,
) -> tuple[bool, object]:
    stage_7_readiness_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS total_queue_rows,
            COUNT(*) FILTER (WHERE queue_status = 'pending') AS pending_count,
            COUNT(*) FILTER (WHERE queue_status = 'claimed') AS claimed_count,
            COUNT(*) FILTER (WHERE queue_status = 'sent') AS sent_count,
            COUNT(*) FILTER (WHERE queue_status = 'failed') AS failed_count,
            COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
            COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS populated_sent_at_count,
            COUNT(*) FILTER (WHERE producer_delivery_error IS NOT NULL) AS populated_error_count
        FROM "{schema}"."{send_queue_table_name}"
        WHERE dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": dataset_id,
            "run_id": run_id,
        },
    )

    row = stage_7_readiness_dataframe.iloc[0]

    ready_for_stage_7 = (
        int(row["total_queue_rows"]) > 0
        and int(row["pending_count"]) == int(row["total_queue_rows"])
        and int(row["claimed_count"]) == 0
        and int(row["sent_count"]) == 0
        and int(row["failed_count"]) == 0
        and int(row["populated_claim_token_count"]) == 0
        and int(row["populated_sent_at_count"]) == 0
        and int(row["populated_error_count"]) == 0
    )

    return ready_for_stage_7, stage_7_readiness_dataframe

----

In [ ]:
# -----------------------------------------------------------------------------
# Check whether queue is clean enough for Stage 7
# -----------------------------------------------------------------------------

ready_for_stage_7, stage_7_readiness_dataframe = check_stage_7_readiness(
    engine=engine,
    schema=SCHEMA,
    send_queue_table_name=SEND_QUEUE_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
)

display(stage_7_readiness_dataframe)

print("Ready for Stage 7:", ready_for_stage_7)

if not ready_for_stage_7:
    print(
        "Queue is not clean for Stage 7. Use the 6.5 reset/repair cells, "
        "then rerun this readiness check."
    )

---

In [ ]:
producer_pickup_explain_dataframe = read_sql_dataframe(
    engine,
    f"""
    EXPLAIN ANALYZE
    SELECT
        message_key,
        observation_index,
        message_sequence_index,
        sensor_index
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
      AND dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT :producer_batch_size
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
        "producer_batch_size": PRODUCER_BATCH_SIZE,
    },
)

display(producer_pickup_explain_dataframe)

,QUERY PLAN
0,Limit (cost=0.56..4286.20 rows=26000 width=76...
1,-> Index Scan using idx_synthetic_sensor_me...
2,Filter: (queue_status = 'pending'::text)
3,Planning Time: 0.323 ms
4,Execution Time: 24.929 ms


---

In [ ]:
# -----------------------------------------------------------------------------
# Claim one producer-sized batch
# -----------------------------------------------------------------------------

claim_token = None
claimed_dataframe = None

if RUN_CLAIM_TEST_FLAG:
    claim_token, claimed_dataframe = claim_pending_sensor_messages_batch(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        batch_size=PRODUCER_BATCH_SIZE,
        producer_worker_id=PRODUCER_WORKER_ID,
        producer_topic=PRODUCER_TOPIC,
    )

    print("Claim token:", claim_token)
    print("Claimed row count:", len(claimed_dataframe))
    print("Expected claimed row count:", PRODUCER_BATCH_SIZE)

    display(claimed_dataframe.head(104))
else:
    print("RUN_CLAIM_TEST_FLAG is False. No rows claimed.")

---

In [ ]:
# -----------------------------------------------------------------------------
# Validate claim status after claim test
# -----------------------------------------------------------------------------

claim_status_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE claimed_at IS NULL) AS null_claimed_at_count,
        COUNT(*) FILTER (WHERE claimed_at IS NOT NULL) AS populated_claimed_at_count,
        COUNT(*) FILTER (WHERE producer_worker_id IS NOT NULL) AS populated_worker_count,
        COUNT(*) FILTER (WHERE producer_topic IS NOT NULL) AS populated_topic_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(claim_status_validation_dataframe)

----

In [ ]:
if claimed_dataframe is None:
    print("No claim test was run.")
elif claimed_dataframe.empty:
    print("No rows were claimed.")
else:
    display(
        claimed_dataframe[
            [
                "claim_token",
                "observation_index",
                "message_sequence_index",
                "sensor_name",
                "sensor_index",
                "queue_status",
                "producer_topic",
                "producer_worker_id",
            ]
        ].head(20)
    )

---

In [ ]:
# -----------------------------------------------------------------------------
# Optional: mark the claimed batch as sent
# -----------------------------------------------------------------------------
# WARNING:
# This changes queue_status to 'sent' and stamps producer_sent_at / producer_ack_at.
# The normal reset-test cell will NOT reset sent rows.
# Only run this when intentionally testing the sent transition.

sent_dataframe = None

if MARK_SENT_TEST_FLAG:
    if not claim_token:
        raise ValueError("No claim_token available. Run the claim test first.")

    sent_dataframe = mark_claimed_batch_sent(
        engine=engine,
        claim_token=claim_token,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked sent row count:", len(sent_dataframe))
    display(sent_dataframe.head(20))
else:
    print("MARK_SENT_TEST_FLAG is False. Skipping mark-sent test.")

In [ ]:
# -----------------------------------------------------------------------------
# Optional: mark the claimed batch as failed
# -----------------------------------------------------------------------------
# WARNING:
# This should not be run after mark-sent on the same claim token.
# It only updates rows still in queue_status = 'claimed'.

failed_dataframe = None

if MARK_FAILED_TEST_FLAG:
    if not claim_token:
        raise ValueError("No claim_token available. Run the claim test first.")

    failed_dataframe = mark_claimed_batch_failed(
        engine=engine,
        claim_token=claim_token,
        error_message=TEST_ERROR_MESSAGE,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked failed row count:", len(failed_dataframe))
    display(failed_dataframe.head(20))
else:
    print("MARK_FAILED_TEST_FLAG is False. Skipping mark-failed test.")

---

In [ ]:
# -----------------------------------------------------------------------------
# Optional: requeue failed messages
# -----------------------------------------------------------------------------
# WARNING:
# The current utility requeues failed rows by table/status and max attempts.
# Keep this flag False unless you intentionally created failed rows in this run.

requeued_failed_dataframe = None

if REQUEUE_FAILED_TEST_FLAG:
    failed_before_requeue_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS failed_candidate_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE queue_status = 'failed'
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    display(failed_before_requeue_dataframe)

    requeued_failed_dataframe = requeue_failed_messages(
        engine=engine,
        max_send_attempts=PRODUCER_MAX_SEND_ATTEMPTS,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Requeued failed row count:", len(requeued_failed_dataframe))
    display(requeued_failed_dataframe.head(20))
else:
    print("REQUEUE_FAILED_TEST_FLAG is False. Skipping failed-message requeue test.")

---

In [ ]:
# -----------------------------------------------------------------------------
# Optional: release stale claims
# -----------------------------------------------------------------------------
# WARNING:
# The current utility releases stale claimed rows by table/status/time.
# Keep this False unless you are intentionally testing stale claim recovery.

released_stale_dataframe = None

if RELEASE_STALE_CLAIMS_TEST_FLAG:
    stale_before_release_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS stale_candidate_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE queue_status = 'claimed'
          AND claimed_at IS NOT NULL
          AND claimed_at < (now() - interval '15 minutes')
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    display(stale_before_release_dataframe)

    released_stale_dataframe = release_stale_claims(
        engine=engine,
        stale_after_minutes=15,
        max_send_attempts=PRODUCER_MAX_SEND_ATTEMPTS,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Released stale row count:", len(released_stale_dataframe))
    display(released_stale_dataframe.head(20))
else:
    print("RELEASE_STALE_CLAIMS_TEST_FLAG is False. Skipping stale-claim release test.")

---

In [ ]:
status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

display(status_dataframe)

---

In [ ]:
# -----------------------------------------------------------------------------
# Reset claimed-but-unsent test rows
# -----------------------------------------------------------------------------
# This resets only rows for this dataset/run that are still claimed and unsent.
# It does not reset rows already marked as sent.

if RESET_TEST_CLAIMS_FLAG:
    before_reset_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            COUNT(*) AS reset_candidate_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    display(before_reset_dataframe)

    execute_sql(
        engine,
        f"""
        UPDATE "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        SET
            queue_status = 'pending',
            claim_token = NULL,
            claimed_at = NULL,
            producer_topic = NULL,
            producer_worker_id = NULL,
            producer_delivery_status = NULL,
            producer_delivery_error = NULL
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Reset claimed-but-unsent test rows.")
else:
    print("RESET_TEST_CLAIMS_FLAG is False. No claimed rows reset.")

---

In [ ]:
# -----------------------------------------------------------------------------
# Post-reset queue validation
# -----------------------------------------------------------------------------

post_reset_status_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY queue_status
    ORDER BY row_count DESC
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(post_reset_status_dataframe)

---

In [ ]:
# -----------------------------------------------------------------------------
# Emergency reset: all unsent claimed rows for this dataset/run
# -----------------------------------------------------------------------------
# WARNING:
# This is broader than the normal test reset.
# Use only if a test stopped after claiming rows and before resetting them.

if EMERGENCY_RESET_ALL_UNSENT_CLAIMS_FLAG:
    emergency_candidate_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT
            queue_status,
            COUNT(*) AS row_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        GROUP BY queue_status
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    display(emergency_candidate_dataframe)

    execute_sql(
        engine,
        f"""
        UPDATE "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        SET
            queue_status = 'pending',
            claim_token = NULL,
            claimed_at = NULL,
            producer_topic = NULL,
            producer_worker_id = NULL,
            producer_delivery_status = NULL,
            producer_delivery_error = NULL
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
          AND dataset_id = :dataset_id
          AND run_id = :run_id
        """,
        params={
            "dataset_id": DATASET_ID,
            "run_id": RUN_ID,
        },
    )

    print("Emergency reset completed.")
else:
    print("EMERGENCY_RESET_ALL_UNSENT_CLAIMS_FLAG is False. Emergency reset skipped.")

----

In [ ]:
# -----------------------------------------------------------------------------
# Check whether queue is clean enough for Stage 7
# -----------------------------------------------------------------------------

ready_for_stage_7, stage_7_readiness_dataframe = check_stage_7_readiness(
    engine=engine,
    schema=SCHEMA,
    send_queue_table_name=SEND_QUEUE_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
)

display(stage_7_readiness_dataframe)

print("Ready for Stage 7:", ready_for_stage_7)

if not ready_for_stage_7:
    print(
        "Queue is not clean for Stage 7. Use the 6.5 reset/repair cells, "
        "then rerun this readiness check."
    )

----

In [ ]:
# Dispose Engine
engine.dispose()